# Tile 级计算：接口模型与常用模式

Tile 计算是 Vector 侧的通用矢量计算方式：以 UB 中的 Tile 为输入输出，一次接口调用处理整个有效区域。它开发效率高，是大多数矢量算子的首选起点。


## 环境准备

本节代码面向 Ascend 950 环境。运行前先完成基础环境配置（详见 01.06）：设置设备号、关闭设备后端自动加载，并完成导入自检。

In [ ]:
import os

os.environ.setdefault('TILE_FWK_DEVICE_ID', '0')
os.environ['TORCH_DEVICE_BACKEND_AUTOLOAD'] = '0'

import pypto_pro.language as pl
import torch
import torch_npu

print('导入完成；设备编译、执行与精度仍需在 950 验证')

## 1. 计算模型：out-first

```python
import pypto_pro.language as pl

with pl.section_vector():
    pl.add(out_tile, lhs_tile, rhs_tile)   # 结果写入 out_tile，不返回新 Tile
```

三条约定：

1. **输出在前**：多数 Tile 计算接口把目的 Tile 作为第一个参数，其后是输入、临时 Tile 与可选参数；
2. **作用于有效区域**：计算作用于 Tile 的有效区域（valid_shape 决定），输入输出的 shape/valid_shape/dtype/layout 需满足接口约束；
3. **原地计算需显式支持**：部分接口允许 out 与某个输入同一 Tile；**未明确说明时不能假定支持原地计算**；
4. **同一流水相邻操作无需同步**：同一 Vector 流水上的相邻计算按依赖顺序执行，操作之间不用插任何同步；只有 Tile 在 MTE2/V/MTE3 等跨流水之间传递时，才需要 02.06 的 auto_mutex 或显式同步。


## 2. 模式一：Tile 与 Tile 逐元素

```python
pl.add(out_tile, lhs_tile, rhs_tile)
pl.mul(out_tile, lhs_tile, rhs_tile)
pl.maximum(out_tile, lhs_tile, rhs_tile)
```

同一标量参与整个有效区域运算的 Tile-Scalar 形式（**并非所有逐元素接口的通用能力**，以 API 文档为准）：

```python
pl.add(out_tile, src_tile, 1.0)
```


## 3. 模式二：原地计算与融合接口

```python
# add 后接 ReLU：原地复用 out_tile，减少中间 Tile
pl.add(out_tile, lhs_tile, rhs_tile)
pl.relu(out_tile, out_tile)
```

还有一步完成"加法+ReLU"的融合矢量接口（见 API 文档"融合矢量计算"）。原地计算省 Tile 但覆盖原数据——后续还要用输入值时，用独立输出 Tile。


## 4. 模式三：比较与选择

```python
pl.eq(mask_tile, lhs_tile, rhs_tile)                     # 生成按位压缩掩码
pl.select(out_tile, mask_tile, lhs_tile, fallback_tile, tmp_tile)  # 按掩码选择
```

掩码/临时 Tile 的类型与 layout 需满足接口要求——这是"掩码分支"的 SIMD 表达（对照第 1 章：SIMT 用真分支，SIMD 用掩码选择）。


## 5. 模式四：归约与广播

```python
pl.sum(out_tile, src_tile, tmp_tile, dim=0)
pl.sum(out_tile, src_tile, tmp_tile, dim=1)
```

**dim 是归约方向，不是删除轴**：源 [M, N] 时 dim=0 对每行沿 N 方向求和 → 输出 [M, 1]；dim=1 对每列沿 M 方向 → 输出 [1, N]。广播则把 [M,1]/[1,N] 结果展开回 [M,N]（如 expand_sub/expand_div，见本节第 8 部分的行均值示例）。


## 6. 模式五：类型转换与转置

```python
pl.cast(out_tile, src_tile, mode=pl.RoundMode.CAST_ROUND)  # 目标类型由 out_tile.dtype 决定
# pl.transpose(...)：交换二维 Tile 两个轴（见 API 文档）
```


## 7. 扩展接口面：六大模式之外还有什么

六大模式是主干，但 tile_computation 全目录还有几类"选择与稀疏"相关的接口，**与第 9/11 章的选择段设计直接相关**——先在此建立索引，用到的章节会展开：

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">类别</th><th style="text-align:left;">代表接口</th><th style="text-align:left;">能力</th><th style="text-align:left;">本课程使用点</th></tr>
<tr><td><b>极值索引归约</b></td><td><code>pl.argmax</code> / <code>pl.argmin</code></td><td>沿维度归约，输出极值的<b>索引</b>；若还需要极值，另做 maximum/minimum 归约</td><td>第 9/11 章"选哪个块/哪一行"的候选实现</td></tr>
<tr><td><b>片上 Gather / Scatter</b></td><td><code>pl.gather</code> / <code>pl.gatherb</code> / <code>pl.gathermask</code> / <code>pl.scatter</code></td><td>按索引 Tile 在片上 Tile 之间重排元素；<b>gather 比较模式</b>可按阈值筛选出满足条件的索引并计数（TopK 筛选用）</td><td>第 9 章选择段替代路径、第 11 章门控选择</td></tr>
<tr><td><b>排序</b></td><td><code>pl.sort32</code></td><td>Tile 内排序（每 32 元素一组）</td><td>第 9 章 top-k 的候选实现</td></tr>
<tr><td><b>复合计算</b></td><td><code>pl.add_relu</code> / <code>pl.add_relu_cast</code> / <code>pl.sub_relu</code> / <code>pl.axpy</code> / <code>pl.fused_mul_add</code> / <code>pl.fused_mul_add_relu</code></td><td>一次调用完成"算术+激活/乘加"融合</td><td>第 5 章 GEMM+Bias+GELU 的 Vector 段（比模式二的原地两连写更省 UB 往返）</td></tr>
<tr><td><b>掩码操作</b></td><td><code>pl.set_mask_norm</code> / <code>pl.set_mask_count</code> / <code>pl.set_vec_mask</code> / <code>pl.reset_mask</code></td><td>设置硬件 Vector mask 的普通/计数模式及掩码值；count 模式表示处理元素数，不是统计比较掩码的 1</td><td>底层掩码控制；与比较输出的压缩 mask Tile 区分，Tile API 通常自动管理硬件掩码</td></tr>
<tr><td><b>字节累计直方图</b></td><td><code>pl.histogram</code></td><td>按源元素<b>字节值</b>生成 256 桶<b>累计</b>计数（基数排序用）</td><td>与第 7 章 SIMT 数值分桶频数是两种语义（07.03 有区分说明）</td></tr>
</table>

> 使用纪律：本表是"知道有、用到再查"的索引，签名与约束以对应 API 文档为准。第 9 章会把"tile gather 比较模式 vs SIMT 门槛仲裁"做正式选型对照。


## 8. 模式组合实战：行归一化的完整推导

六大模式很少单独出现——以"每行减均值"（x - mean(x)，[8, 256] FP32）为例走一遍组合过程：

**第一步：归约求均值**（模式四）

```python
# sum(dim=0) 对 [8,256] 归约 → [8, 1]
pl.sum(mean_dn, x_tile, tmp_tile, dim=0)
# mean_dn 为 [8,1] DN；mean_rm 为同地址的 [1,8] 行主序视图
# Tile-Scalar 在行主序视图上原地计算，结果仍可从 DN 视图读取
pl.div(mean_rm, mean_rm, 256.0)
```

**第二步：广播相减**（模式一 + 广播）

```python
# mean_dn 是 [8,1]，x 是 [8,256]，使用按行广播接口
pl.expand_sub(out_tile, x_tile, mean_dn, dim=0)
```

**为什么归约接口自带 dim 而广播要单独做？** 归约的输出形状由 dim 唯一确定（[M,N]→[M,1] 或 [1,N]），接口可以"顺便"知道方向；广播的输入可能来自任何地方（Host 传入、外部常量），方向不唯一——所以归约族接口带 dim、广播族接口也带 dim（expand_*），两者成对使用。

**陷阱清单**（这个组合的三个高频错误）：

1. 归约方向搞反（dim=0/1 记混）→ 输出 [8,1] vs [1,256]，归约语义不同——**先写注释确认方向**再写代码；
2. 广播维度不匹配（例如把 [1,256] 当作 dim=0 的行统计量）→ 编译报 shape 不匹配——检查归约输出的形状；
3. tmp_tile 忘记规划（sum 接口需要临时 Tile）→ 地址重叠或未分配。FP32 行归约/按行广播使用 [8,1] DN 视图；除法等逐元素操作使用同地址 [1,8] 行主序视图，不能把所有操作直接串在 DN Tile 上。视图共享同一组统计量，并非再次分配独立数据；仍须规划物理对齐与生命周期。上面只是计算序列，未包含这些 Tile 的声明和同步；PyPTO 源码仓库的回归测试 `python/tests/st/pypto_pro/frontend/tile_vector/test_layernorm.py` 里，red_type/red_rm_type 正是这两个视图的完整声明，可对照阅读。


### 完整主例：从平方和到 RMSNorm

本例固定 x/y 为 [8,256] FP32，省略 gamma。平方、归约临时量各占 8192 B；8 个统计量占 32 B。`stat_dn` 与 `stat_rm` 绑定同一个 0x8000 地址，是同一份统计量的两种视图：前者供归约和按行广播，后者供 div/add/rsqrt。这个重叠有意且有顺序，不是额外分配两份数据。

数据按“平方 → 行和 → 除 256 → 加 eps → rsqrt → 按行乘”推进。每次操作后可以写出统计量含义：sum(x²)、mean(x²)、mean(x²)+eps、倒数标准差。把这些含义与代码变量对应，比记住接口顺序更容易发现漏除 N 的错误。


In [ ]:
@pl.jit(auto_mutex=True)
def rms_tile(x: pl.Tensor[[8, 256], pl.DT_FP32],
             y: pl.Tensor[[8, 256], pl.DT_FP32]):
    tt = pl.TileType(shape=[8, 256], dtype=pl.DT_FP32, target_memory=pl.MemorySpace.Vec)
    dn = pl.TileType(shape=[8, 1], dtype=pl.DT_FP32, target_memory=pl.MemorySpace.Vec, layout=pl.DN)
    rm = pl.TileType(shape=[1, 8], dtype=pl.DT_FP32, target_memory=pl.MemorySpace.Vec)
    gx = pl.make_tile_group(type=tt, addrs=0, mutex_ids=[0])
    gy = pl.make_tile_group(type=tt, addrs=0x2000, mutex_ids=[1])
    gs = pl.make_tile_group(type=tt, addrs=0x4000, mutex_ids=[2])
    gw = pl.make_tile_group(type=tt, addrs=0x6000, mutex_ids=[3])
    gd = pl.make_tile_group(type=dn, addrs=0x8000, mutex_ids=[4])
    gr = pl.make_tile_group(type=rm, addrs=0x8000, mutex_ids=[5])
    with pl.section_vector():
        tx = gx.current()
        ty = gy.current()
        sq = gs.current()
        tmp = gw.current()
        stat_dn = gd.current()
        stat_rm = gr.current()
        pl.load(tx, x, [0, 0])
        pl.mul(sq, tx, tx)                    # 平方
        pl.sum(stat_dn, sq, tmp, dim=0)       # 行平方和 -> [8,1] DN
        pl.div(stat_rm, stat_rm, 256.0)       # 行主序视图上除 N
        pl.add(stat_rm, stat_rm, 1e-5)        # 加 eps
        pl.rsqrt(stat_rm, stat_rm)            # 倒数标准差
        pl.expand_mul(ty, tx, stat_dn, dim=0) # 按行广播缩放
        pl.store(y, ty, [0, 0])


In [ ]:
device = f"npu:{int(os.environ['TILE_FWK_DEVICE_ID'])}"
torch.npu.set_device(device)
torch.manual_seed(42)
x = torch.randn(8, 256, device=device, dtype=torch.float32)
y = torch.full_like(x, float('nan'))
rms_tile[None, 1](x, y)
torch.npu.synchronize()
golden = x / torch.sqrt((x * x).mean(-1, keepdim=True) + 1e-5)
assert torch.isfinite(y).all().item()
torch.testing.assert_close(y, golden, rtol=1e-4, atol=1e-5)
print('Tile RMSNorm 校验通过')


## 9. 课后练习

1. （填空）`pl.sum(out, src, tmp, dim=0)` 对 [8, 256] 的源 Tile 归约，输出 shape 为 ______。
2. （选择）接口文档未说明支持原地计算时，`pl.add(t, a, t)`（out 与输入同 Tile）：
   A. 一定可以<br/>B. 不能假定可行，需查文档或实测确认
3. （实践）用"归约 + 广播 + 逐元素"三步组合出 `x - mean(x)`（每行减去行均值），写出接口调用序列（不求一次写对，写出你的推导）。

In [ ]:
!cat ./answer/03.03_answer.txt

## 10. 本节小结

- Tile 计算 out-first、作用于有效区域、原地计算需文档背书；
- 六大模式：**逐元素 / Tile-Scalar / 原地与融合 / 比较选择 / 归约广播 / 类型转换与转置**；
- 归约 dim 是方向（[M,N]：dim=0 → [M,1]，dim=1 → [1,N]）。

下一节回答：什么时候放弃 Tile 计算、改用 Reg 计算。